# NorthStar Urban Mobility — Part 2: R Analytics
**Module:** Databases and Analytics  
**Section:** R Analytics — Statistical Analysis, Data Manipulation, Visualisation (15 marks)

This notebook applies R-based statistical analysis and visualisation to uncover operational patterns in the NorthStar dataset, including delivery performance drivers, vehicle fleet health, customer satisfaction correlates, and complaint dynamics.

In [ ]:
# Install and load packages
pkgs <- c("ggplot2", "dplyr", "tidyr", "lubridate", "corrplot", "scales", "ggcorrplot")
for (p in pkgs) {
  if (!require(p, character.only = TRUE))
    install.packages(p, repos = "https://cran.r-project.org")
  library(p, character.only = TRUE)
}
cat("All packages loaded\n")

In [ ]:
# Load datasets
customers  <- read.csv("customers.csv",  stringsAsFactors = FALSE)
orders     <- read.csv("orders.csv",     stringsAsFactors = FALSE)
deliveries <- read.csv("deliveries.csv", stringsAsFactors = FALSE)
drivers    <- read.csv("drivers.csv",    stringsAsFactors = FALSE)
vehicles   <- read.csv("vehicles.csv",   stringsAsFactors = FALSE)
complaints <- read.csv("complaints.csv", stringsAsFactors = FALSE)
incidents  <- read.csv("incidents.csv",  stringsAsFactors = FALSE)

# ── Zone normalisation (same as Part 1) ──────────────────────────────────────
norm_zone <- function(x) {
  x <- trimws(toupper(x))
  dplyr::case_when(
    x %in% c("CTR","CENTRAL")         ~ "Central",
    x == "NORTH"                       ~ "North",
    x == "SOUTH"                       ~ "South",
    x == "EAST"                        ~ "East",
    x == "WEST"                        ~ "West",
    x == "AIRPORT"                     ~ "Airport",
    x %in% c("RIVERSIDE","RIVERSIDE")  ~ "Riverside",
    TRUE                               ~ x
  )
}
customers$home_zone    <- norm_zone(customers$home_zone)
drivers$base_zone      <- norm_zone(drivers$base_zone)
vehicles$assigned_zone <- norm_zone(vehicles$assigned_zone)
orders$pickup_zone     <- norm_zone(orders$pickup_zone)
orders$dropoff_zone    <- norm_zone(orders$dropoff_zone)

# Parse datetimes
deliveries$dispatch_time         <- ymd_hms(deliveries$dispatch_time,         quiet=TRUE)
deliveries$delivery_completed_at <- ymd_hms(deliveries$delivery_completed_at, quiet=TRUE)
orders$order_created_at          <- ymd_hms(orders$order_created_at,          quiet=TRUE)

# Compute actual delivery duration (hours)
deliveries$actual_hours <- as.numeric(
  difftime(deliveries$delivery_completed_at, deliveries$dispatch_time, units="hours")
)

cat("Data loaded and prepared\n")

## 2.1 Descriptive Statistics — Core Operational Variables

In [ ]:
# Descriptive summary of key numeric fields
cat("=== Delivery numeric summary ===\n")
delivery_vars <- deliveries[, c("route_distance_km","manual_route_override_count",
                                 "customer_rating_post_delivery","fuel_or_charge_cost",
                                 "actual_hours")]
print(summary(delivery_vars))

cat("\n=== Driver numeric summary ===\n")
print(summary(drivers[, c("years_experience","training_score","driver_rating")]))

cat("\n=== Vehicle numeric summary ===\n")
print(summary(vehicles[, c("battery_health_pct","odometer_km")]))

## 2.2 Distribution Analysis — Customer Rating by Delivery Status

In [ ]:
ggplot(deliveries %>% filter(!is.na(customer_rating_post_delivery)),
       aes(x = delivery_status, y = customer_rating_post_delivery, fill = delivery_status)) +
  geom_violin(trim = FALSE, alpha = 0.7) +
  geom_boxplot(width = 0.15, fill = "white", outlier.colour = "#c0392b", outlier.size = 1.5) +
  scale_fill_manual(values = c("OnTime"="#27ae60", "Delayed"="#f39c12", "Failed"="#c0392b")) +
  labs(
    title    = "Customer Rating Distribution by Delivery Status",
    subtitle = "Violin + boxplot overlay; red dots = outliers",
    x = "Delivery Status", y = "Customer Rating (1–5)"
  ) +
  theme_minimal(base_size = 13) +
  theme(legend.position = "none")

## 2.3 Correlation Analysis — Delivery and Driver Variables

In [ ]:
# Join deliveries with drivers and orders for a combined analytical frame
combined <- deliveries %>%
  left_join(orders,   by = "order_id") %>%
  left_join(drivers,  by = "driver_id") %>%
  left_join(vehicles, by = "vehicle_id")

# Select numeric columns for correlation
cor_vars <- combined %>%
  select(customer_rating_post_delivery, route_distance_km,
         manual_route_override_count, fuel_or_charge_cost,
         actual_hours, driver_rating, training_score,
         years_experience, battery_health_pct, odometer_km) %>%
  drop_na()

cor_matrix <- cor(cor_vars, use = "complete.obs")

ggcorrplot(cor_matrix,
           method   = "circle",
           type     = "lower",
           lab      = TRUE,
           lab_size = 2.8,
           colors   = c("#c0392b", "white", "#27ae60"),
           title    = "Correlation Matrix: Delivery, Driver and Vehicle Variables",
           ggtheme  = theme_minimal(base_size = 11))

**Interpretation:** Strong positive correlation between driver_rating and customer_rating_post_delivery confirms that driver quality directly influences customer satisfaction. Negative correlation between manual_route_override_count and customer_rating suggests overrides degrade experience. Battery health and odometer_km are negatively correlated, indicating that high-mileage EVs suffer accelerated battery degradation.

## 2.4 Time Series — Monthly Delivery Volume and Failure Rate

In [ ]:
# Monthly trend of delivery outcomes
monthly_trend <- deliveries %>%
  filter(!is.na(dispatch_time)) %>%
  mutate(month = floor_date(dispatch_time, "month")) %>%
  group_by(month) %>%
  summarise(
    total      = n(),
    failed     = sum(delivery_status == "Failed"),
    delayed    = sum(delivery_status == "Delayed"),
    ontime     = sum(delivery_status == "OnTime"),
    fail_rate  = round(100 * (failed + delayed) / total, 2),
    .groups    = "drop"
  )

# Plot
ggplot(monthly_trend, aes(x = month)) +
  geom_col(aes(y = total), fill = "#bdc3c7", alpha = 0.6) +
  geom_line(aes(y = fail_rate * max(total) / 100), colour = "#c0392b", linewidth = 1.2) +
  geom_point(aes(y = fail_rate * max(total) / 100), colour = "#c0392b", size = 2.5) +
  scale_y_continuous(
    name   = "Delivery Volume",
    sec.axis = sec_axis(~ . * 100 / max(monthly_trend$total), name = "Failure+Delay Rate (%)")
  ) +
  labs(
    title    = "Monthly Delivery Volume vs Failure/Delay Rate",
    subtitle = "Bars = total volume; Red line = combined failure + delay rate",
    x = "Month"
  ) +
  theme_minimal(base_size = 13)

## 2.5 Statistical Test — Does Driver Employment Type Affect Delivery Outcome?

In [ ]:
# Chi-squared test: employment type vs delivery status
delivery_driver <- deliveries %>%
  left_join(drivers, by = "driver_id") %>%
  filter(!is.na(employment_type), !is.na(delivery_status))

contingency_table <- table(delivery_driver$employment_type, delivery_driver$delivery_status)
cat("Contingency Table: Employment Type vs Delivery Status\n")
print(contingency_table)

chi_result <- chisq.test(contingency_table)
cat(sprintf("\nChi-squared = %.3f, df = %d, p-value = %.4f\n",
            chi_result$statistic, chi_result$parameter, chi_result$p.value))

if (chi_result$p.value < 0.05) {
  cat("Conclusion: Significant association between employment type and delivery outcome (p < 0.05).\n")
} else {
  cat("Conclusion: No statistically significant association detected.\n")
}

In [ ]:
# Proportional stacked bar: delivery status by employment type
delivery_driver %>%
  group_by(employment_type, delivery_status) %>%
  summarise(n = n(), .groups = "drop") %>%
  group_by(employment_type) %>%
  mutate(pct = 100 * n / sum(n)) %>%
  ggplot(aes(x = employment_type, y = pct, fill = delivery_status)) +
  geom_col(position = "fill") +
  scale_y_continuous(labels = percent_format(scale = 100)) +
  scale_fill_manual(values = c("OnTime"="#27ae60", "Delayed"="#f39c12", "Failed"="#c0392b")) +
  labs(
    title = "Delivery Outcome Proportion by Driver Employment Type",
    x = "Employment Type", y = "Proportion", fill = "Status"
  ) +
  theme_minimal(base_size = 13)

## 2.6 Fleet Health Analysis — Battery Degradation by Vehicle Type

In [ ]:
vehicles_clean <- vehicles %>% filter(!is.na(battery_health_pct))

# Boxplot of battery health by vehicle type and maintenance status
ggplot(vehicles_clean, aes(x = vehicle_type, y = battery_health_pct, fill = maintenance_status)) +
  geom_boxplot(outlier.colour = "#c0392b", alpha = 0.8) +
  geom_hline(yintercept = 60, linetype = "dashed", colour = "#c0392b") +
  annotate("text", x = 0.6, y = 62, label = "Risk threshold (60%)", size = 3.5, colour = "#c0392b") +
  scale_fill_manual(values = c("Active"="#27ae60", "InRepair"="#e67e22", "Scheduled"="#3498db")) +
  labs(
    title    = "Battery Health % by Vehicle Type and Maintenance Status",
    subtitle = "Dashed line = 60% risk threshold",
    x = "Vehicle Type", y = "Battery Health (%)", fill = "Maintenance Status"
  ) +
  theme_minimal(base_size = 13)

## 2.7 Complaint Severity and Resolution Time Analysis

In [ ]:
# Average resolution days by complaint type and severity
complaint_summary <- complaints %>%
  group_by(complaint_type, severity) %>%
  summarise(
    count             = n(),
    avg_resolution    = mean(resolution_days, na.rm = TRUE),
    avg_compensation  = mean(compensation_amount, na.rm = TRUE),
    .groups           = "drop"
  )

# Heatmap: complaint type vs severity -> avg resolution days
complaint_summary %>%
  ggplot(aes(x = severity, y = complaint_type, fill = avg_resolution)) +
  geom_tile(colour = "white", linewidth = 0.5) +
  geom_text(aes(label = round(avg_resolution, 1)), size = 3.5, colour = "#2c3e50") +
  scale_fill_gradient(low = "#f9e79f", high = "#c0392b") +
  labs(
    title = "Average Resolution Days: Complaint Type × Severity",
    x = "Severity", y = "Complaint Type", fill = "Avg Days"
  ) +
  theme_minimal(base_size = 13)

## 2.8 Summary of R Analytics Findings

| Analysis | Key Insight |
|----------|-------------|
| Rating distributions | Failed deliveries produce significantly lower ratings with higher variance |
| Correlation matrix | Driver rating and training score are the strongest positive predictors of customer satisfaction |
| Monthly trends | Failure rates exhibit seasonal variation; certain months show spikes warranting further investigation |
| Chi-squared test | Employment type is a statistically significant predictor of delivery outcome |
| Fleet health | A subset of EVs fall below the 60% battery threshold while classified as 'Active' — a latent maintenance risk |
| Complaint resolution | High-severity Billing and DriverBehaviour complaints take the longest to resolve, driving compensation costs |